# proto-REINFORCE, notebook 2: Last-Minute Pricing

This morning you sold the seats of a small plane in the arcade: 5 seats, 4 days to takeoff, one price a day,
PREMIUM at \$5 a seat or STANDARD at \$2. In notebook 1, proto-REINFORCE learned to manage Old Bessie. Here the same
algorithm learns to price the plane, and you write only the three pieces that change: the row of a situation
(task A), one flight (task B) and the loop (task C). Your functions of notebook 1 come back finished.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing or the finished code
of notebook 1: run it, and open it only if you are curious. The 💡 hints are folded cells too: double-click one to
open it, and open the next one only if you are still stuck. It takes about 20 to 25 minutes.

Start with the three folded cells below: the plane's kit, the checks, and your functions of notebook 1.

In [ ]:
#@title The plane's kit (run me first) { display-mode: "form" }
# Last-Minute Pricing's kit. Run this cell first; you do not need to read it.
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

SEATS = 5                       # the seats of the plane
DAYS = 4                        # the days to takeoff: DAY 1, 2, 3 and 4
PRICES = ['PREMIUM', 'STANDARD']
PREMIUM, STANDARD = 0, 1                # the columns of a table: the prices
DOLLARS = [5, 2]                  # a seat costs $5 at PREMIUM and $2 at STANDARD
FLIGHTS = 2500                  # the flights of every average
# The rows of a table: every situation a flight can be in, [seats left, day]. STANDARD sells at most 3 seats a
# day, so on DAY 2 at least 2 seats are left; on DAY 3 and DAY 4 anything from 1 to 5.
SITUATIONS = [[5, 1], [5, 2], [4, 2], [3, 2], [2, 2], [5, 3], [4, 3], [3, 3], [2, 3], [1, 3], [5, 4], [4, 4], [3, 4], [2, 4], [1, 4]]

# The plane of the arcade (levers.js and pricing.js): how many buyers come in a day, with their chances
_KIT_BUYERS = [[[0, 0.6], [1, 0.4]], [[1, 0.2], [2, 0.4], [3, 0.4]]]
_KIT_START_BETS = [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]
_KIT_BEST_LINE = 11.8392       # the best strategy's average money on the flights of update 240
_KIT_BEST = [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 2]      # the best price of each row (2: either)
_KIT_LOSES = [0.0256, 0.224, 0.064, 0.96, 2.016, 2.08, 1.12, 0.16, 0.96, 1.2, 2.4, 2.4, 2.4, 1.6, 0.0]      # what the other price loses there, on average
_KIT_OFTEN = [1.0, 0.6, 0.4, 0.0, 0.0, 0.0, 0.36, 0.4, 0.24, 0.0, 0.0, 0.0, 0.312, 0.448, 0.24]      # the share of the best strategy's flights that are in that row


def sell(seats_left, price, u):
    """One day of the plane. With this many seats left and this price, the number u (between 0 and 1) picks how
    many buyers come, with the game's chances; they buy what is left. Gives the seats sold and the day's money:
        seats_sold, money = sell(5, STANDARD, 0.5)"""
    if isinstance(seats_left, bool) or seats_left not in (1, 2, 3, 4, 5) or isinstance(price, bool) or \
            price not in (0, 1):
        raise ValueError("sell needs 1 to 5 seats left and a price, 0 for PREMIUM or 1 for STANDARD; it got %r "
                         "seats left and the price %r" % (seats_left, price))
    running = 0
    buyers = _KIT_BUYERS[price][-1][0]
    for count, chance in _KIT_BUYERS[price]:
        running = running + chance           # the running chances, added as pricing.js adds them
        if u < running:
            buyers = count
            break
    seats_sold = min(buyers, seats_left)
    return seats_sold, DOLLARS[price] * seats_sold


def roll_dice(seed, flights):
    """The dice of many flights: dice[flight][day] holds two numbers between 0 and 1, the first to pick the price,
    the second for the buyers. The same seed always rolls the same numbers."""
    state = seed & 0xFFFFFFFF
    dice = []
    for f in range(flights):
        flight = []
        for d in range(DAYS):
            pair = []
            for k in range(2):
                state = (state + 0x6D2B79F5) & 0xFFFFFFFF
                t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
                t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
                pair.append(((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296)
            flight.append(pair)
        dice.append(flight)
    return dice


_KIT_DICE = {}


def dice_of_update(number):
    """The 2,500 flights of an update: seed 12 for the start, update 0, and for updates 1 and 2; fresh
    flights (seed 2000 + number) for every later update."""
    seed = 12 if number <= 2 else 2000 + number
    if seed not in _KIT_DICE:
        if len(_KIT_DICE) >= 2:
            _KIT_DICE.clear()
        _KIT_DICE[seed] = roll_dice(seed, FLIGHTS)
    return _KIT_DICE[seed]


def _kit_numbers(seed):
    state = [seed & 0xFFFFFFFF]

    def next_number():
        state[0] = (state[0] + 0x6D2B79F5) & 0xFFFFFFFF
        s = state[0]
        t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296
    return next_number


_KIT_PICKS = []


def picked_bet(number):
    """The bet that update number (1, 2, 3, ...) picks at random: [row, price]."""
    if isinstance(number, bool) or not isinstance(number, int) or number < 1:
        raise ValueError("picked_bet needs the number of an update: 1, 2, 3 and so on; it got %r" % (number,))
    if len(_KIT_PICKS) < number:
        next_number = _kit_numbers(134623)
        del _KIT_PICKS[:]
        while len(_KIT_PICKS) < number:
            k = int(next_number() * 2 * len(SITUATIONS))   # one of the 30 bets, each as likely as the others
            _KIT_PICKS.append([k // 2, k % 2])             # its row, the situation, and its column, the price
    return list(_KIT_PICKS[number - 1])


def start_bets():
    """The starting table of bets, picked at random between -10 and 10: a fresh copy each time."""
    return copy_bets(_KIT_START_BETS)


def copy_bets(bets):
    """A copy of a table of bets: change the copy, and the table stays as it was."""
    copy = []
    for row in bets:
        copy.append(list(row))
    return copy


def _kit_situation(row):
    """DAY 1, 5 SEATS LEFT"""
    seats_left, day = SITUATIONS[row]
    return "DAY %d, %d %s LEFT" % (day, seats_left, "SEAT" if seats_left == 1 else "SEATS")


def show_situations():
    """Print the situations a flight can be in, one per row of a table."""
    print("row   situation")
    for row in range(len(SITUATIONS)):
        print("%3d   %s" % (row, _kit_situation(row)))


def _kit_is_table(table):
    if not isinstance(table, list) or len(table) != len(SITUATIONS):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != 2:
            return False
        for x in row:
            if isinstance(x, bool) or not isinstance(x, (int, float)) or x != x:
                return False
    return True


def _kit_short(value):
    text = repr(value)
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, mark):
    head = "row   situation           " + "%10s%10s" % (PRICES[0], PRICES[1])
    if mark:
        head += "    likeliest"
    print(head)
    for row in range(len(SITUATIONS)):
        line = "%3d   %-20s" % (row, _kit_situation(row)) + "%10.2f%10.2f" % (table[row][0], table[row][1])
        if mark:
            line += "    " + PRICES[0 if table[row][0] > table[row][1] else 1]
        print(line)


def show_bets(bets):
    """Print a table of bets: a row per situation, a column per price, two decimals."""
    if not _kit_is_table(bets):
        print("⏳ No table of bets to show yet: show_bets needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(bets)))
        return
    _kit_print_table(bets, False)


def show_chances(table):
    """Print a table of chances (the policy): a row per situation, a column per price, two decimals, and the
    likeliest price of each row."""
    if not _kit_is_table(table):
        print("⏳ No table of chances to show yet: show_chances needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(table)))
        return
    _kit_print_table(table, True)


def _kit_money(x):
    """$10.18"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def plot_rewards(points, best=_KIT_BEST_LINE, label="best strategy"):
    """The reward plot, as text: across, the number of updates; up, the average money of a flight after them. o is
    a point (one column for every few updates when there are many), and the row of dashes marks the best strategy's
    average money on the flights of the last update."""
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if isinstance(x, bool) or not isinstance(x, (int, float)) or x != x:
                ok = False
    if not ok:
        print("⏳ No reward plot yet: plot_rewards needs a list of at least two averages. It got %s." % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 0.1
    for size in (0.1, 0.2, 0.25, 0.5, 1, 2, 5, 10):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(round(high / step, 9)) * step
    bottom = _kit_math.floor(round(low / step, 9)) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The reward plot: the average money of a flight after each update" +
          (" (one o for every %d updates)" % every if every > 1 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_money(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 100 if n >= 300 else (50 if n >= 200 else (10 if n >= 40 else 1))
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))


def show_best(bets):
    """The best strategy, row by row, worked out backwards from DAY 4 with the game's chances, beside the likeliest
    price of a table of bets."""
    if not _kit_is_table(bets):
        print("⏳ No table of bets to compare yet: show_best needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(bets)))
        return
    print("best:  the best price in this situation, with the best prices after it")
    print("loses: what the other price loses here, on average, in dollars")
    print("often: the share of the best strategy's flights that are ever in this situation")
    print("yours: the likeliest price of your table")
    print()
    print("row   situation             best       loses   often   yours")
    same, matters = 0, 0
    for row in range(len(SITUATIONS)):
        yours = 0 if bets[row][0] > bets[row][1] else 1
        best = "either" if _KIT_BEST[row] == 2 else PRICES[_KIT_BEST[row]]
        mark = ""
        if _KIT_BEST[row] != 2:
            matters += 1
            if yours == _KIT_BEST[row]:
                same += 1
                mark = "  ✓"
            else:
                mark = "  ✗"
        print("%3d   %-20s  %-9s %6s   %4d%%   %-9s%s" % (row, _kit_situation(row), best, _kit_money(_KIT_LOSES[row]),
                                                       round(100 * _KIT_OFTEN[row]), PRICES[yours], mark))
    print()
    print("Your table's likeliest price is the best one in %d of the %d situations where the price matters." % (
        same, matters))

In [ ]:
#@title The checks of notebook 2 (run me too) { display-mode: "form" }
# The checks of notebook 2 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N2_TESTS = {'row_of': [{'about': '5 seats left on DAY 1', 'args': [5, 1], 'kind': 'action', 'want': 0, 'tol': 0}, {'about': '2 seats left on DAY 2', 'args': [2, 2], 'kind': 'action', 'want': 4, 'tol': 0}, {'about': '4 seats left on DAY 3', 'args': [4, 3], 'kind': 'action', 'want': 6, 'tol': 0}, {'about': '3 seats left on DAY 4', 'args': [3, 4], 'kind': 'action', 'want': 12, 'tol': 0}, {'about': '1 seats left on DAY 4', 'args': [1, 4], 'kind': 'action', 'want': 14, 'tol': 0}, {'about': '5 seats left on DAY 3', 'args': [5, 3], 'kind': 'action', 'want': 5, 'tol': 0}], 'play_flight': [{'about': 'flight 1 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 0]}], 'kind': 'number', 'want': 11, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 2 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 1]}], 'kind': 'number', 'want': 16, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 3 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 2]}], 'kind': 'number', 'want': 16, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 4 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 3]}], 'kind': 'number', 'want': 2, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 5 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 4]}], 'kind': 'number', 'want': 14, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 6 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 5]}], 'kind': 'number', 'want': 4, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 7 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 6]}], 'kind': 'number', 'want': 14, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 8 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 7]}], 'kind': 'number', 'want': 2, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 9 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 8]}], 'kind': 'number', 'want': 9, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 10 of the start (the bets at random)', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.574442516811659, 0.425557483188341], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 9]}], 'kind': 'number', 'want': 16, 'tol': 1e-09, 'keep': [0]}, {'about': 'flight 1 after the update of PREMIUM on DAY 4 with 4 seats left', 'args': [[[0.9998989708060922, 0.00010102919390777277], [0.6899744811276125, 0.31002551887238755], [0.21416501695744136, 0.7858349830425586], [0.3775406687981454, 0.6224593312018546], [0.28905049737499605, 0.710949502625004], [0.9999924954984029, 7.504501597110005e-06], [0.5249791874789399, 0.47502081252106004], [0.9983411989198256, 0.0016588010801744215], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106004], [0.005486298899450406, 0.9945137011005496], [0.05215356307841773, 0.9478464369215822], [2.2544429650445423e-05, 0.9999774555703497], [0.7685247834990178, 0.23147521650098235], [0.3775406687981454, 0.6224593312018546]], {'run': [12, 0]}], 'kind': 'number', 'want': 11, 'tol': 1e-09}, {'about': 'flight 1 with a table of chances that always plays PREMIUM', 'args': [[[1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0]], {'run': [12, 0]}], 'kind': 'number', 'want': 10, 'tol': 1e-09}, {'about': 'flight 1 with a table of chances that always plays STANDARD', 'args': [[[0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1]], {'run': [12, 0]}], 'kind': 'number', 'want': 10, 'tol': 1e-09}], 'proto_reinforce': [{'about': 'the first 7 updates', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 7, 0.01, 20], 'kind': 'numbers', 'want': [10.2596, 10.2596, 10.2596, 10.35, 10.3776, 10.3732, 10.3996, 10.1984], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, -2.4], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}}, {'about': 'the first 7 updates with alpha = 25', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 7, 0.01, 25], 'kind': 'numbers', 'want': [10.2596, 10.2596, 10.2596, 10.35, 10.3776, 10.3732, 10.3996, 10.2036], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, -3.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}}]}
_N2_SLIPS = {'row_of': [{'answers': [0, 4, 2, 3, 9, 0], 'hint': 'a situation matches when both numbers match: situation[0] == seats_left and situation[1] == day'}, {'answers': [1, 5, 7, 13, 15, 6], 'hint': 'count from 0: the first row, DAY 1 with 5 seats left, is row 0'}], 'play_flight': [{'answers': [0, None, None, 0, None, 0, 0, 0, 0, None, 0, 0, None], 'hint': 'seats_left goes down by the seats sold: seats_left - seats_sold'}, {'answers': [11, 13, 13, 2, 14, 4, 10, 4, 5, 13, 11, 10, 10], 'hint': "numbers[0] picks the price and numbers[1] is the buyers' luck, the u of sell"}], 'proto_reinforce': [{'answers': [[10.2596, 10.2596, 10.2596, 10.35, 10.3776, 10.3732, 10.3996, 10.2028], [10.2596, 10.2596, 10.2596, 10.35, 10.3776, 10.3732, 10.3996, 10.2004]], 'hint': "the slope of an update plays this update's flights: dice, from dice_of_update(number)"}, {'answers': [[10.2596, 10.2596, 10.2596, 10.3532, 10.3776, 10.3736, 10.4028, 10.2128], [10.2596, 10.2596, 10.2596, 10.3532, 10.3776, 10.3736, 10.4028, 10.2128]], 'hint': 'slope measures with the hop: slope(bets, row, price, dice, hop), and alpha goes to update_bet'}]}
_N2_SUCCESS = {'row_of': 'row_of works: DAY 1 with 5 seats left is row 0, DAY 4 with 1 seat left row 14.', 'play_flight': 'play_flight works: the first ten flights of the start bring $11, $16, $16, $2, $14, $4, $14, $2, $9, $16.', 'proto_reinforce': 'proto_reinforce works: its first 7 updates end at the point $10.1984, and update 7 moves a bet.'}
_N2_DEFAULT = {'row_of': 'compare both numbers of the situation: situation[0] == seats_left and situation[1] == day; give back row, the counter, and count from 0', 'play_flight': 'the row is row_of(seats_left, day), the price is pick_action(policy[row], numbers[0]), and seats_left goes down by seats_sold', 'proto_reinforce': 'nabla = slope(bets, row, price, dice, hop), then update_bet(bets, row, price, nabla, alpha)'}
_N2_KEEP_HINT = {'row_of': 'row_of reads SITUATIONS and leaves it as it is', 'play_flight': 'play_flight reads the table of chances and leaves it as it is', 'proto_reinforce': 'proto_reinforce changes the table of bets it gets, one bet per update, with update_bet(bets, row, price, nabla, alpha)'}
_N2_TASKS = {'row_of': 'task A', 'play_flight': 'task B', 'proto_reinforce': 'task C'}
_N2_NEEDS = {'row_of': [], 'chances': [], 'pick_action': [], 'update_bet': [], 'table_of_chances': ['chances'], 'play_flight': ['row_of', 'pick_action'], 'average_money': ['table_of_chances', 'play_flight'], 'slope': ['average_money'], 'proto_reinforce': ['average_money', 'slope', 'update_bet']}
_N2_LONG = ['proto_reinforce']
_N2_NONE_HINT = {'row_of': 'row_of gives None when no situation matches: situation[0] is the seats left and situation[1] the day, as in [5, 1]'}
_N2_DICE_OF = 'roll_dice'
_N2_GAPS = {}
_N2_WORKING = {}
_N2_DICE = {}


def _n2_num(x):
    """A number with at most 4 decimals: 154.3961, 2.14, 174."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n2_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n2_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n2_num(value)


def _n2_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n2_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n2_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n2_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n2_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n2_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n2_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n2_like(got, answer, tol):
    """True when got has the shape and the numbers of a slip's answer, within tol: a slip may give a number where a
    list is due (return total), or a list of numbers where a list of lists is due."""
    if isinstance(answer, (list, tuple)):
        if not isinstance(got, (list, tuple)) or len(got) != len(answer):
            return False
        for k in range(len(answer)):
            if not _n2_like(got[k], answer[k], tol):
                return False
        return True
    return _n2_is_number(got) and _n2_is_number(answer) and abs(got - answer) <= tol


def _n2_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n2_copy(x))
        return out
    return value


def _n2_dice(seed, first, count):
    """The dice of count runs of a seed from run first, rolled by the kit once per seed and kept."""
    rolled = _N2_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N2_DICE) > 3:
            _N2_DICE.clear()
        rolled = globals()[_N2_DICE_OF](seed, max(first + count, 10))
        _N2_DICE[seed] = rolled
    return rolled[first:first + count]


def _n2_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n2_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "run" in a:
            args.append(_n2_dice(a["run"][0], a["run"][1], 1)[0])
        else:
            args.append(_n2_copy(a))
    return args


def _n2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N2_GAPS:
        return _N2_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N2_GAPS[code] = gap
    return gap


def _n2_holds_gap(value):
    """True when a value, or a list inside it, still holds a ... (a gap of a cell above)."""
    if value is Ellipsis:
        return True
    if isinstance(value, (list, tuple)):
        for x in value:
            if _n2_holds_gap(x):
                return True
    return False


def _n2_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n2_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N2_NEEDS.get(name, []):
        key.extend(_n2_key(other, seen + (name,)))
    return tuple(key)


def _n2_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n2_call.__code__.co_filename, globals()[_N2_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n2_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back. row_of gives None for a situation that is not in SITUATIONS: call it as row_of(seats_left, day)'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a row or a price, such as policy[row] for the row of a situation and row_of_chances[price] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = (update_bet "
                "changes the table itself and gives back only the new bet)")
    if 'sell needs' in text:
        return 'the plane got a number of seats or a price it does not know: check what pick_action gives back, and that seats_left goes down by the seats sold'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n2_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint). The kit's state (its luck) is put back
    afterwards when the kit can save it."""
    save = globals().get("_kit_save")
    saved = save() if callable(save) else None
    try:
        return _n2_failure_of(name, fn)
    finally:
        if saved is not None:
            globals()["_kit_restore"](saved)


def _n2_failure_of(name, fn):
    tests = _N2_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n2_args(test)
        kept = _n2_copy(args)
        for setup in test.get("setup", []):
            globals()[setup[0]](*setup[1])
        run = _n2_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n2_explain(run))
        got = run["answer"]
        if not _n2_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N2_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N2_DEFAULT[name]
            for slip in _N2_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n2_like(got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n2_short(test["want"]),
                                                                    _n2_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n2_short(kept[i])), _N2_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n2_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of bets should be %s, and it is %s" % (
                    name, _n2_short(t["want"]), _n2_short(args[t["arg"]])), _N2_KEEP_HINT[name])
    return None


def _n2_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n2_has_gap(fn):
        return False
    for other in _N2_NEEDS.get(name, []):
        if _n2_uses(fn, other) and not _n2_works(other):
            return False
    if name not in _N2_TESTS:
        return True
    key = _n2_key(name)
    if key not in _N2_WORKING:
        _N2_WORKING[key] = _n2_failure(name, fn) is None
    return _N2_WORKING[key]


def _n2_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N2_TASKS:
        task = " (%s)" % _N2_TASKS[name]
    if _n2_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N2_NEEDS.get(name, []):
        if not _n2_uses(fn, other) or _n2_works(other):
            continue
        if other in _N2_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n2_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N2_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N2_NEEDS.get(other, []):
            if deeper in _N2_TASKS and not _n2_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N2_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n2_waits(name, fn, "this check"):
        return
    failure = _n2_failure(name, fn)
    if failure is None:
        print("✅ " + _N2_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n2_safe_check(name, fn):
    try:
        _n2_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_row_of(fn):
    """Task a: the rows of several situations, the first and the last among them"""
    _n2_safe_check("row_of", fn)

def check_play_flight(fn):
    """Task b: the MONEY of the first ten flights of the start, of a flight after one update, and of a flight that always plays PREMIUM or always STANDARD"""
    _n2_safe_check("play_flight", fn)

def check_proto_reinforce(fn):
    """Task c: the points and the table of the first 7 updates, with alpha 20 and with alpha 25"""
    _n2_safe_check("proto_reinforce", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_money, bets, dice) is
    average_money(bets, dice). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_money, bets, dice).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n2_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
            if _n2_holds_gap(x):
                print("⏳ %s waits for a value that still holds a ...: fill it in above, run its cell, then this cell."
                      % name)
                return None
        if name in _N2_LONG and not _n2_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        long = name in _N2_LONG
        if long and callable(globals().get("_kit_before_long")):
            globals()["_kit_before_long"]()
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        finally:
            if long and callable(globals().get("_kit_after_long")):
                globals()["_kit_after_long"]()
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

def _n2_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N2_TASKS[name], cell))
        return False
    if _n2_waits(name, fn, cell):
        return False
    if not _n2_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def show_flight(policy, flight_dice):
    """Plays one flight with your play_flight and prints it day by day: the situation, the price, the seats sold and
    the money, then how the flight ends and its MONEY."""
    try:
        if policy is None or flight_dice is None:
            print("⏳ show_flight waits for a table of chances from a cell above (its ⏳ line says why).")
            return
        if not _n2_ready("play_flight", "this cell"):
            return
        log = []
        the_sell = globals()["sell"]

        def sell_written_down(seats_left, price, u):
            seats_sold, money = the_sell(seats_left, price, u)
            log.append([seats_left, price, seats_sold, money])
            return seats_sold, money
        globals()["sell"] = sell_written_down
        try:
            total = play_flight(policy, flight_dice)
        finally:
            globals()["sell"] = the_sell
        left = SEATS
        for k in range(len(log)):
            seats_left, price, seats_sold, money = log[k]
            print("DAY %d   %d %-10s  %-8s  %d %-10s  $%d" % (
                k + 1, seats_left, "SEAT LEFT" if seats_left == 1 else "SEATS LEFT", PRICES[price], seats_sold,
                "seat sold" if seats_sold == 1 else "seats sold", money))
            left = seats_left - seats_sold
        if left == 0:
            print("FULL FLIGHT!   MONEY $%d" % total)
        else:
            print("TAKEOFF! %d EMPTY %s.   MONEY $%d" % (left, "SEAT" if left == 1 else "SEATS", total))
    except Exception as error:
        print("❌ show_flight could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of notebook 2 are ready.")

In [ ]:
#@title From notebook 1: chances, table_of_chances, pick_action and update_bet (run me) { display-mode: "form" }
# From notebook 1, finished: softmax, the table of chances, the roulette and the update. On the plane an action is
# a price, 0 for PREMIUM and 1 for STANDARD, and a state is the row of a situation.

def chances(row_of_bets):
    # row_of_bets: the bets of one row, one per price. Gives the chances: e to the power of each bet, each divided by
    # their total.
    powers = []
    for bet in row_of_bets:
        powers.append(exp(bet))         # e to the power of the bet: always above 0
    total = 0
    for power in powers:
        total = total + power           # the total of the powers
    row_of_chances = []
    for power in powers:
        row_of_chances.append(power / total)    # the chances add up to 1
    return row_of_chances


def table_of_chances(bets):
    # bets: the table of bets, one row per situation. Gives the table of chances, one row per situation.
    table = []
    for row_of_bets in bets:
        table.append(chances(row_of_bets))      # softmax, one row at a time
    return table


def pick_action(row_of_chances, u):
    # row_of_chances: the chances of one row. u: a number between 0 and 1.
    # Gives the action, here the price, whose slice u falls in: 0 for PREMIUM, 1 for STANDARD.
    running = 0                                 # the chances added up so far
    for action in range(len(row_of_chances)):
        running = running + row_of_chances[action]      # the end of this action's slice
        if u < running:                         # u falls in this action's slice
            return action
    return len(row_of_chances) - 1              # u at the very end of the line: the last action


def update_bet(bets, state, action, nabla, alpha):
    # The update of one bet, w = w + alpha nabla, in cents. Changes the table of bets and gives the new bet.
    w = bets[state][action]                     # the bet as it is
    new_bet = w + alpha * nabla                 # a positive nabla raises the bet, a negative one lowers it
    bets[state][action] = round(new_bet, 2)     # into the table, in cents
    return bets[state][action]


print("✅ From notebook 1: chances, table_of_chances, pick_action and update_bet are ready.")

## The plane, one day at a time

Every day you set one price for the seats that are left:

- **PREMIUM**, \$5 a seat: no buyer on 60% of the days, one buyer on 40%.
- **STANDARD**, \$2 a seat: one buyer on 20% of the days, two on 40%, three on 40%.

The buyers take the seats that are left. A flight ends with a FULL FLIGHT when the last seat is sold, or at
takeoff after DAY 4, with its EMPTY SEATS. Its MONEY is the plain sum of its days: no discount, since a flight
lasts only 4 days. The game never shows these chances; the kit knows them.

The kit plays a day for you: `sell(seats_left, price, u)` gives the seats sold and the day's money, and the number
`u`, between 0 and 1, decides how many buyers come. Prices are numbers, the columns of a table: `PREMIUM` is 0,
`STANDARD` 1.

In [ ]:
# One day of the plane: with the seats left and a price, the number u decides how many buyers come
seats_sold, money = sell(5, PREMIUM, 0.3)
print("5 seats left, PREMIUM, u = 0.3:    seats sold", seats_sold, "   money", money)

seats_sold, money = sell(2, STANDARD, 0.9)
print("2 seats left, STANDARD, u = 0.9:   seats sold", seats_sold, "   money", money)

**Reading the result.** At PREMIUM, `u` = 0.3 falls in the slice of no buyer, the first 60% of the numbers: no
seat sold. At STANDARD, `u` = 0.9 falls in the slice of three buyers, the last 40%, but only 2 seats are left: 2
seats sold, \$4.

## The dice

As in notebook 1, every average plays the same flights: `dice[flight][day]` holds two numbers between 0 and 1.
The first picks the price, the second is the `u` of `sell`, the buyers' luck. With the same dice, two tables of
bets meet the same buyers day for day. When the plane is full, the numbers of its later days are not used.

In [ ]:
# The dice of the start: 2,500 flights of 4 days, two numbers a day
dice = dice_of_update(0)
print(len(dice), "flights of", len(dice[0]), "days")
print("DAY 1 of flight 1:", dice[0][0])

## The situations

Old Bessie's strategy looked at the van's state. The plane's price depends on two things: the seats left and
the day. A situation is a pair of numbers, `[seats left, day]`, and every flight starts in `[5, 1]`: 5 seats left
on DAY 1.

💭 **Think first.** Which situations can a flight be in on DAY 2? And on DAY 3?

In [ ]:
show_situations()

**Reading the result.** At STANDARD at most 3 seats sell in a day, at PREMIUM at most 1. So on DAY 2 the plane has 5, 4, 3 or 2
seats left, and on DAY 3 and DAY 4 anything from 1 to 5: 15 situations. The arcade's board has 20 squares; the
other 5 are squares no flight ever reaches, and they get no row. The kit keeps the situations in `SITUATIONS`,
in this order: `SITUATIONS[0]` is `[5, 1]`.

## 🎯 Task A. The row of a situation

A strategy reads one row of the table: the row of today's situation. Write `row_of(seats_left, day)`: go through
`SITUATIONS` with a counter, and give back the row of the situation whose seats left and day are these.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# situation is a pair such as [4, 3]: situation[0] is its seats left, 4, and situation[1] its day, 3. The if
# asks whether both match, joined by and.

In [ ]:
#@title 💡 Hint 2 for task A: the line (double-click to open) { display-mode: "form" }
#         if situation[0] == seats_left and situation[1] == day:

In [ ]:
# 🎯 YOUR TURN, task A: the row of a situation
def row_of(seats_left, day):
    # Gives the row of the table for a situation: the place of [seats_left, day] in SITUATIONS.
    row = 0                                     # the row of the situation we look at
    for situation in SITUATIONS:                # situation[0]: its seats left, situation[1]: its day
        if ...:                                 # 🎯 YOUR TURN: this situation has these seats left and this day
            return row
        row = row + 1

In [ ]:
check_row_of(row_of)

## The table of bets

One row per situation and one column per price: 30 bets, picked at random between -10 and 10, with a few
extreme ones. Softmax turns each row into chances, with your `chances` of notebook 1: the table of chances is the
strategy.

In [ ]:
bets = start_bets()                             # the bets at random
show_bets(bets)
print()
policy = play(table_of_chances, bets)           # the table of chances
show_chances(policy)

**Reading the result.** Bets far apart make one price almost certain: on DAY 3 with 5 seats left, 6.80 on PREMIUM and -5.00 on STANDARD give PREMIUM a
chance of 1.00, rounded. Bets close together play both prices: on DAY 3 with 4 seats left, PREMIUM gets 0.52 and STANDARD 0.48.

## 🎯 Task B. One flight

A flight is a run of at most 4 days. Each day: the row of the situation (task A), the roulette of that row with
the day's first number (`pick_action`, from notebook 1), then `sell` with the second number. The money adds up
plainly, and the seats sold are gone.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The row is row_of(seats_left, day). The price is the roulette of policy[row], the row of chances of today's
# situation, spun with numbers[0]. After the day, seats_left goes down by seats_sold.

In [ ]:
#@title 💡 Hint 2 for task B: the three lines (double-click to open) { display-mode: "form" }
#             row = row_of(seats_left, day)
#             price = pick_action(policy[row], numbers[0])
#             seats_left = seats_left - seats_sold

In [ ]:
# 🎯 YOUR TURN, task B: one flight
def play_flight(policy, flight_dice):
    # policy: the table of chances. flight_dice: the dice of one flight, two numbers a day.
    # Plays the flight from DAY 1 with 5 seats left and gives its MONEY: the plain sum of its days.
    seats_left = SEATS                          # nobody has bought a seat yet
    day = 1
    total = 0
    for numbers in flight_dice:                 # one day: numbers[0] picks the price, numbers[1] is the buyers' luck
        if seats_left > 0:                      # a FULL FLIGHT sells nothing more
            row = ...                           # 🎯 YOUR TURN: the row of today's situation (task A)
            price = ...                         # 🎯 YOUR TURN: the roulette of that row, with numbers[0]
            seats_sold, money = sell(seats_left, price, numbers[1])     # the day happens
            total = total + money               # no discount: a dollar on DAY 4 counts as much as on DAY 1
            seats_left = ...                    # 🎯 YOUR TURN: the seats sold today are gone
        day = day + 1
    return total

In [ ]:
check_play_flight(play_flight)

💭 **Think first.** Flight 1 of the dice, with the bets at random. Day by day, what does the strategy do?

In [ ]:
show_flight(policy, dice[0])

**Reading the result.** The strategy plays PREMIUM, PREMIUM, PREMIUM and STANDARD, and the flight ends with 1 EMPTY SEAT, MONEY \$11. Each day the price comes
from the row of that day's situation.

## The average and the slope

From notebook 1, with one change. `average_money` averages the MONEY of the 2,500 flights, and `slope` measures the
hop of one bet on the same flights. In notebook 1 both were in cents. A hop of 0.01 changes the price of only a few
flights in 2,500, by a few dollars each, so the average moves by a fraction of a cent: in cents, all 240 slopes of
the loop at the end would be 0. Here every digit is kept.

In [ ]:
# From notebook 1, with one change: every digit is kept
def average_money(bets, dice):
    # bets: the table of bets. dice: the dice of many flights.
    # Gives the average MONEY of the flights, every digit kept.
    policy = table_of_chances(bets)             # softmax of every row
    total = 0
    for flight_dice in dice:                    # one flight
        total = total + play_flight(policy, flight_dice)
    return total / len(dice)                    # the change: no rounding to cents


def slope(bets, row, price, dice, hop):
    # How fast the average MONEY rises when the bet of row and price rises, measured with a hop.
    before = average_money(bets, dice)          # Before: the table as it is
    hopped = copy_bets(bets)                    # a copy: the table itself stays as it is
    hopped[row][price] = hopped[row][price] + hop          # the bet, one hop higher
    after = average_money(hopped, dice)         # After: the same flights, with the hopped table
    change = after - before                     # the change: every digit, no rounding to cents
    nabla = change / hop                        # up divided by across
    return round(nabla, 2)                      # nabla, to the hundredth


print("✅ average_money and slope are ready.")

## One bet by hand

Take the bet of PREMIUM on DAY 4 with 4 seats left. The bets at random make PREMIUM the likelier price there,
with a chance of 0.57.

💭 **Think first.** On the last day, PREMIUM sells at most one seat and STANDARD one to three. Will a hop up of
this bet raise the average MONEY or lower it? And how many of the 2,500 flights do you expect to change?

In [ ]:
# One bet by hand: PREMIUM on DAY 4 with 4 seats left
hop = 0.01                                      # the hop: a small change of one bet, as on page 3
bets = start_bets()
row, price = 11, PREMIUM                        # row 11 is DAY 4 with 4 seats left (show_situations above)

hopped = copy_bets(bets)                        # a copy: the table itself stays as it is
hopped[row][price] = hopped[row][price] + hop   # one hop up

before = play(average_money, bets, dice)        # Before: the table as it is
after = play(average_money, hopped, dice)       # After: the same flights, with the hopped table
nabla = play(slope, bets, row, price, dice, hop)
print("Before:  the bet", round(bets[row][price], 2), "   the average MONEY", before)
print("After:   the bet", round(hopped[row][price], 2), "   the average MONEY", after)
print("nabla =", nabla)

**Reading the result.** Lower. Only 1 of the 2,500 flights plays another price after the hop, and the average moves from \$10.2596 to \$10.258: a change of -\$0.0016,
so

$$\nabla = \frac{-0.0016}{0.01} = -0.16$$

A small number: the money of a flight is a few dollars, and a hop changes few flights. In notebook 1, nabla was
174.

## alpha

💭 **Think first.** In notebook 1, alpha was 0.01. With this nabla, how far would alpha 0.01 move the bet?

In [ ]:
# The update of that bet, then the same 2,500 flights again
alpha = 20                                      # alpha: turns a small slope into a move of a few units
bets = start_bets()
new_bet = play(update_bet, bets, row, price, nabla, alpha)
print("The new bet:", new_bet)
print("The average MONEY after the update:", play(average_money, bets, dice))

**Reading the result.** alpha 0.01 would move the bet by 0.16 times 0.01, that is 0.0016: nothing to see. So on the plane
alpha is 20, which turns this slope into a move of 3.2:

$$w = 0.10 + 20 \times (-0.16) = -3.10$$

PREMIUM's chance on DAY 4 with 4 seats left falls from 0.57 to 0.05, and the average MONEY of the same flights goes
from \$10.2596 to \$10.494.

## 🎯 Task C. proto-REINFORCE on the plane

The loop of notebook 1, with the plane's functions: a bet picked at random, the flights of that update, the slope
of the bet, its update, and one more point for the reward plot. `picked_bet(number)` gives a row and a price, one
of the 30 bets.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The slope is slope(bets, row, price, dice, hop), with this update's dice.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# The update changes the table itself, so the line is the call alone:
#         update_bet(bets, row, price, nabla, alpha)

In [ ]:
# 🎯 YOUR TURN, task C: proto-REINFORCE on the plane
def proto_reinforce(bets, updates, hop, alpha):
    # bets: the table of bets, changed by every update. updates: how many updates.
    # Gives the points of the reward plot: the average MONEY before the first update, then after each update.
    points = [average_money(bets, dice_of_update(0))]      # the first point: the bets as they are
    for number in range(1, updates + 1):                   # update 1, 2, 3, ...
        row, price = picked_bet(number)                    # the bet picked at random
        dice = dice_of_update(number)                      # the flights of this update
        nabla = ...                                        # 🎯 YOUR TURN: the slope of that bet
        ...                                                # 🎯 YOUR TURN: the update of that bet
        points.append(average_money(bets, dice))           # one more point: the average MONEY after the update
    return points

In [ ]:
check_proto_reinforce(proto_reinforce)

## 240 updates

💭 **Think first.** On DAY 4, the seats that are not sold fly empty. The bets at random play PREMIUM on DAY 4 with 2 seats left (a
chance of 0.77) and STANDARD on DAY 3 with 2 seats left (0.60). Which price do you expect there after the updates?

The next cell plays 240 updates, each with three averages of 2,500 flights: about half a minute on Colab. There are 30 bets, so each
gets about 8 updates.

In [ ]:
# 240 updates of proto-REINFORCE on the plane, from the bets at random (about half a minute)
bets = start_bets()
points = play(proto_reinforce, bets, 240, hop, alpha)

In [ ]:
plot_rewards(points)

**Reading the result.** The first point is the start, \$10.26, and after update 240 the average is \$11.77. The row of
dashes is the best strategy's average MONEY on the flights of update 240, \$11.84. In 210 of the 240 updates nabla was 0:
the hop changed the money of no flight. A hop of 0.01 moves a chance by at most 0.0025, by less when the row's
chances are near 0 and 1, and a rare row has few flights to change. So the points climb in steps, and from update
3 on every update plays fresh flights, which makes them wobble.

In [ ]:
show_chances(play(table_of_chances, bets))

## The best strategy

The kit works the best strategy out exactly, backwards from DAY 4 with the game's chances. It is in no table of
bets: proto-REINFORCE never sees it.

In [ ]:
show_best(bets)

**Reading the result.** The best strategy expects \$11.77 a flight. In the four situations where flights often go
and the price matters most, your table now plays the best price: STANDARD on DAY 4 with 3 or 2 seats left, since
an empty seat earns nothing at takeoff, and on DAY 3 with 4 seats left; PREMIUM on DAY 3 with 2 seats left, since
two days remain to find buyers at \$5. It still differs from the best strategy on DAY 2 with 3 seats left, DAY 2 with 2 seats left, DAY 3 with 5 seats left and DAY 4 with 4 seats left, situations its flights almost never reach (fewer than 1 in 1,000): proto-REINFORCE learns where its own flights go. It also differs on DAY 2 with 4 seats left, where the other price loses only \$0.06. Your table expects \$11.73 a flight, up from \$10.35 at the start.

## What you built

- **The same algorithm on a new game**: the forward pass plays flights with your `play_flight`, the backward pass
  hops one bet and measures the slope, the update moves it by alpha times the slope.
- **What changed**: a strategy reads the row of a situation, `row_of(seats_left, day)`; the MONEY of a flight is a
  plain sum; every digit of the average is kept; alpha is 20.

Notebook 1, Old Bessie: [proto-REINFORCE, notebook 1: Old Bessie](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_1_old_bessie_student.ipynb).